# FOLIO Detailed Error Analysis

This notebook investigates specific error patterns:
1. Cases where Lean axiomatizes conclusions
2. Cases where Two-Stage fails but simple Lean succeeds
3. Cases where both Lean approaches fail

All analysis grounded in actual result files.

In [ ]:
import json
import pandas as pd
import re

# Load results
with open('../results/folio/cot/all_results.json', 'r') as f:
    cot_results = json.load(f)

with open('../results/folio/lean/all_results.json', 'r') as f:
    lean_results = json.load(f)

with open('../results/folio/two_stage/all_results.json', 'r') as f:
    two_stage_results = json.load(f)

# Flatten CoT
cot_flat = []
for story in cot_results:
    if 'results' in story:
        for result in story['results']:
            cot_flat.append(result)

cot_sorted = sorted(cot_flat, key=lambda x: x.get('example_id'))
lean_sorted = sorted(lean_results, key=lambda x: x.get('example_id'))
two_stage_sorted = sorted(two_stage_results, key=lambda x: x.get('example_id'))

print(f"Loaded {len(lean_sorted)} questions")

## 1. Find Cases Where Lean Axiomatizes Conclusions

Identify verified but wrong cases and check for axiomatization patterns.

In [ ]:
# Find all verified but wrong cases
verified_wrong_cases = []

for lean in lean_sorted:
    if (lean.get('lean_verification', {}).get('success', False) and 
        not lean['correct']):
        verified_wrong_cases.append(lean)

print(f"Found {len(verified_wrong_cases)} verified but wrong cases")
print("\nAnalyzing axiomatization patterns...\n")

# Analyze first 10 cases
for i, case in enumerate(verified_wrong_cases[:10]):
    lean_code = case.get('lean_code', '')
    
    # Count suspicious axioms (not just type declarations or entity declarations)
    suspicious_axioms = []
    for line in lean_code.split('\n'):
        if 'axiom' in line.lower():
            # Skip type and entity declarations
            if not any(x in line for x in [': Type', ': Entity', 'Entity :']):
                suspicious_axioms.append(line.strip())
    
    print(f"Case {i+1}: Example ID {case.get('example_id')}")
    print(f"  Ground truth: {case.get('ground_truth')}")
    print(f"  Prediction: {case.get('prediction')}")
    print(f"  Suspicious axioms: {len(suspicious_axioms)}")
    if suspicious_axioms:
        print(f"  First 3: ")
        for ax in suspicious_axioms[:3]:
            print(f"    {ax[:80]}..." if len(ax) > 80 else f"    {ax}")
    print()

## 2. Two-Stage Fails but Lean Succeeds

Investigate cases where simple Lean gets the right answer but Two-Stage doesn't.

In [ ]:
lean_wins = []

for lean, ts in zip(lean_sorted, two_stage_sorted):
    if lean['correct'] and not ts['correct']:
        lean_wins.append({
            'example_id': lean.get('example_id'),
            'ground_truth': lean.get('ground_truth'),
            'lean_pred': lean.get('prediction'),
            'ts_pred': ts.get('prediction'),
            'lean_verified': lean.get('lean_verification', {}).get('success', False),
            'ts_verified': ts.get('all_cycles', [{}])[-1].get('lean_success', False) if ts.get('all_cycles') else False,
            'ts_cycles': len(ts.get('all_cycles', []))
        })

print(f"Found {len(lean_wins)} cases where Lean succeeds but Two-Stage fails")
print("\nFirst 5 cases:\n")

df = pd.DataFrame(lean_wins[:5])
print(df.to_string(index=False))

## 3. Both Lean Approaches Fail

Cases where both simple Lean and Two-Stage get wrong answers.

In [ ]:
both_wrong = []

for cot, lean, ts in zip(cot_sorted, lean_sorted, two_stage_sorted):
    if not lean['correct'] and not ts['correct']:
        both_wrong.append({
            'example_id': lean.get('example_id'),
            'ground_truth': lean.get('ground_truth'),
            'cot_correct': cot['correct'],
            'lean_pred': lean.get('prediction'),
            'ts_pred': ts.get('prediction'),
            'lean_verified': lean.get('lean_verification', {}).get('success', False),
            'ts_verified': ts.get('all_cycles', [{}])[-1].get('lean_success', False) if ts.get('all_cycles') else False
        })

print(f"Found {len(both_wrong)} cases where both Lean approaches fail")
print(f"Of these, CoT is correct in {sum(1 for case in both_wrong if case['cot_correct'])} cases")
print("\nBreakdown by verification status:\n")

both_verified = sum(1 for case in both_wrong if case['lean_verified'] and case['ts_verified'])
neither_verified = sum(1 for case in both_wrong if not case['lean_verified'] and not case['ts_verified'])
mixed = len(both_wrong) - both_verified - neither_verified

print(f"Both verified but wrong: {both_verified}")
print(f"Neither verified: {neither_verified}")
print(f"Mixed (one verified, one not): {mixed}")

## 4. Detailed Case Study: Example 2

Deep dive into the Joey example showing axiomatization problem.

In [ ]:
# Get example 2 from all three approaches
ex2_cot = next(r for r in cot_sorted if r.get('example_id') == 2)
ex2_lean = next(r for r in lean_sorted if r.get('example_id') == 2)
ex2_ts = next(r for r in two_stage_sorted if r.get('example_id') == 2)

print("Example 2: Joey the Turkey")
print("="*70)
print(f"\nConclusion: {ex2_cot.get('conclusion')}")
print(f"Ground Truth: {ex2_cot.get('ground_truth')}")
print(f"\nCoT:       {ex2_cot.get('prediction')} {'(Correct)' if ex2_cot['correct'] else '(Wrong)'}")
print(f"Lean:      {ex2_lean.get('prediction')} {'(Correct)' if ex2_lean['correct'] else '(Wrong)'} - Verified: {ex2_lean.get('lean_verification', {}).get('success', False)}")
print(f"Two-Stage: {ex2_ts.get('prediction')} {'(Correct)' if ex2_ts['correct'] else '(Wrong)'} - Verified: {ex2_ts.get('all_cycles', [{}])[-1].get('lean_success', False) if ex2_ts.get('all_cycles') else False}")

print("\n" + "="*70)
print("LEAN CODE - Key Lines:")
print("="*70)
lean_code = ex2_lean.get('lean_code', '')
for i, line in enumerate(lean_code.split('\n'), 1):
    # Show lines with Joey
    if 'Joey' in line:
        marker = ' <- AXIOMATIZES CONCLUSION' if ('axiom' in line.lower() and ('Wild' in line or 'Turkey' in line)) else ''
        print(f"{i:3d}: {line}{marker}")

## 5. Summary Statistics

In [ ]:
print("="*70)
print("FOLIO ERROR ANALYSIS SUMMARY")
print("="*70)

print(f"\nTotal questions: {len(lean_sorted)}")
print(f"\n1. Axiomatization Problem:")
print(f"   Verified but wrong (Lean): {len(verified_wrong_cases)} ({len(verified_wrong_cases)/len(lean_sorted)*100:.1f}%)")

print(f"\n2. Two-Stage vs Lean Disagreements:")
print(f"   Lean correct, Two-Stage wrong: {len(lean_wins)}")
print(f"   Both wrong: {len(both_wrong)}")

print(f"\n3. CoT Advantage:")
cot_rescues = sum(1 for case in both_wrong if case['cot_correct'])
print(f"   Cases where only CoT is correct: {cot_rescues}")
print(f"   These are cases where formalization itself causes errors")

print("="*70)